Se comienza creando el enterno virtual, en este caso llamado .venv, y se activan los scripts.

Al comenzar el desarrollo, se preparan las librerias a utilizar: pandas, numpy, seaborn,matplotlib y el kernel con ipykernel para evitar problemas al momento de correr el enterno virtual.

In [ ]:
# Importación de librerías esenciales
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de gráficos
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [10, 6]

# Carga del dataset
df = pd.read_csv("spotify_data_clean.csv")

# Inspección de dimensiones y tipos
print(f"Dimensiones del dataset: {df.shape[0]} filas y {df.shape[1]} columnas.\n")
print("--- Estructura y Tipos de Datos ---")
df.info()

In [ ]:
df.head()

# Tratamiento de Valores Nulos
Al inspeccionar el dataset, descubrimos que las variables numéricas y objetivo están completas. Solo encontramos nulos en variables de texto:
1. `artist_name` (3 nulos): Al ser un volumen insignificante, optamos por eliminar esos registros.
2. `artist_genres` (3361 nulos): Para no perder la valiosa información de miles de filas, imputamos estos valores vacios con la categoría "Desconocido".

In [ ]:
# Eliminar filas donde el nombre del artista es nulo
# (Usamos df_limpio para mantener el original intacto)
df_limpio = df.dropna(subset=['artist_name']).copy()

# 2. Rellenar los nulos de los géneros musicales con "Desconocido"
df_limpio['artist_genres'] = df_limpio['artist_genres'].fillna('Desconocido')

# 3. Verificación de nulos
print("--- Control final de nulos ---")
print(df_limpio.isnull().sum())

# Filtrado de Ruido y Duplicados
Una vez que el dataset está completo (sin nulos), eliminamos registros que no aportan valor o son repetidos.

In [ ]:
# Filtrar el ruido (Popularidad 0)
cantidad_original = len(df_limpio)
df_limpio = df_limpio[df_limpio['track_popularity'] > 0].copy()
print(f"Se eliminaron {cantidad_original - len(df_limpio)} canciones con popularidad 0.")

# Verificar duplicados (por ID)
duplicados_id = df_limpio.duplicated(subset=['track_id']).sum()
print(f"Canciones duplicadas por 'track_id': {duplicados_id}")

# Si hay, los eliminamos
if duplicados_id > 0:
    df_limpio = df_limpio.drop_duplicates(subset=['track_id'], keep='first')
    print("¡Duplicados eliminados exitosamente!")

In [ ]:
df_limpio["track_popularity"].unique()

# Transformación y Exportación
Esta es la celda técnica donde preparamos el "lenguaje" que entenderá el Random Forest.

In [ ]:
df['album_type'].unique()

In [ ]:
# Transformar 'explicit' a binario
mapeo_explicito = {True: 1, False: 0, 'TRUE': 1, 'FALSE': 0, 'True': 1, 'False': 0}
df_limpio['explicit'] = df_limpio['explicit'].map(mapeo_explicito)

# Aplicar One-Hot Encoding a 'album_type'
# Convertimos categorías a columnas numéricas (0 y 1)
# df_limpio = pd.get_dummies(df_limpio, columns=['album_type'], drop_first=True, dtype=int)

df_limpio['album_type'] = df['album_type'].map({
    'album':1,
    'single':2,
    'compilation':3
    })


# Verificación final de la estructura
print("\n--- Estructura final de columnas ---")
print(df_limpio.dtypes)

# Exportación
archivo_salida = 'spotify_datos_preprocesados.csv'
df_limpio.to_csv(archivo_salida, index=False)
print(f"\n¡Procesamiento completo! Archivo guardado como: {archivo_salida}")

In [ ]:
df_limpio['album_type'].unique()